### Prepare environment

In [ ]:
%run ../../environment/prepare_environment_llm

# Notebook 9.4 - Agent Tools with MCP

Connect to the workshop MCP server hosted as a Databricks App and call tools that expose module metadata and document retrieval.

MCP gives agents a standard way to discover and call tools. In this notebook, an LLM receives the discovered MCP tool descriptions, selects one tool for the user message, and the notebook executes that tool call.

## 1. Configure MCP Client

The Databricks App exposes an MCP endpoint. The notebook connects to that endpoint, lists available tools, and calls them just like an agent runtime would.

Databricks Apps require audience-scoped OAuth tokens for notebook-to-app calls. The notebook gets the app OAuth client ID from the Databricks SDK, exchanges the notebook token for an app-scoped token, and passes that token as an HTTP authorization header. The token is not printed.

In [ ]:
import json
import re
from typing import Any

import requests
from databricks.sdk import WorkspaceClient
from mlflow.deployments import get_deploy_client
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


def notebook_api_token() -> str:
    """Read the short-lived API token from the Databricks notebook context.

    The token identifies the current notebook user inside the workspace. It is
    not the final token used for the app request; Databricks Apps require an
    audience-scoped OAuth token, so this value becomes the subject token in the
    token exchange flow.
    """
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    try:
        token = context.apiToken().get()
    except Exception as exc:
        raise ValueError(
            "The notebook context did not expose an API token. "
            "Run this notebook inside Databricks."
        ) from exc
    if not token:
        raise ValueError("The notebook context returned an empty API token.")
    return token


def exchange_notebook_token_for_app_token(app_name: str) -> str:
    """Exchange the notebook token for a token scoped to one Databricks App.

    A Databricks App checks the `audience` claim of incoming OAuth tokens. The
    notebook token is valid for workspace APIs, but it does not have the app as
    its audience. The Databricks OAuth token-exchange endpoint returns a new
    access token whose audience is the app OAuth client ID.
    """
    workspace = WorkspaceClient()
    app = workspace.apps.get(app_name)
    app_client_id = app.oauth2_app_client_id
    if not app_client_id:
        raise ValueError(
            f"Databricks App {app_name!r} does not expose oauth2_app_client_id."
        )

    token_url = f"{workspace.config.host.rstrip('/')}/oidc/v1/token"
    response = requests.post(
        token_url,
        data={
            "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
            "subject_token": notebook_api_token(),
            "subject_token_type": "urn:databricks:params:oauth:token-type:personal-access-token",
            "requested_token_type": "urn:ietf:params:oauth:token-type:access_token",
            "scope": "all-apis",
            "audience": app_client_id,
        },
        timeout=30,
    )
    if not response.ok:
        raise RuntimeError(
            f"Token exchange failed with HTTP {response.status_code}: {response.text}"
        )

    payload = response.json()
    access_token = payload.get("access_token")
    if not access_token:
        raise RuntimeError(
            f"Token exchange response did not contain access_token: {payload}"
        )
    return access_token


def databricks_app_headers(app_name: str) -> dict[str, str]:
    """Create HTTP headers accepted by a protected Databricks App."""
    app_token = exchange_notebook_token_for_app_token(app_name)
    return {"Authorization": f"Bearer {app_token}"}


dbutils.widgets.text("mcp_app_name", "mcp-genai-workshop")
dbutils.widgets.text("mcp_app_url", "")
dbutils.widgets.text("llm_endpoint_name", "databricks-meta-llama-3-3-70b-instruct")

mcp_app_name = dbutils.widgets.get("mcp_app_name").strip()
mcp_app_url = dbutils.widgets.get("mcp_app_url").strip().rstrip("/")
llm_endpoint_name = dbutils.widgets.get("llm_endpoint_name")

if not mcp_app_name:
    raise ValueError("Set the mcp_app_name widget to the Databricks App name.")
if not mcp_app_url:
    raise ValueError(
        "Set the mcp_app_url widget to the Databricks App URL before running this notebook."
    )

mcp_url = mcp_app_url if mcp_app_url.endswith("/mcp") else f"{mcp_app_url}/mcp"
mcp_headers = databricks_app_headers(mcp_app_name)
deploy_client = get_deploy_client("databricks")
print(
    {
        "mcp_app_name": mcp_app_name,
        "mcp_url": mcp_url,
        "llm_endpoint_name": llm_endpoint_name,
        "authenticated": True,
    }
)

## 2. Define MCP Utility Functions

MCP responses can contain structured content or text content. These helpers normalize the response into Python objects so the rest of the notebook can display tool results cleanly.

In [ ]:
def as_jsonable(value: Any) -> Any:
    """Convert SDK or Pydantic objects into JSON-friendly values.

    MCP clients can return rich Python objects, but notebooks and agent traces
    are easiest to inspect as plain strings, numbers, lists, and dictionaries.
    This helper removes that SDK-specific shape before results are displayed or
    passed to the next step.
    """
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool, list, tuple, dict)):
        return value
    if hasattr(value, "model_dump"):
        return value.model_dump()
    if hasattr(value, "dict"):
        return value.dict()
    if hasattr(value, "__dict__"):
        return value.__dict__
    return str(value)


def parse_tool_result(result: Any) -> Any:
    """Extract the useful payload from an MCP tool call result.

    MCP tools may return structured content, text content, or SDK wrapper
    objects. An agent loop should work with the actual payload, not the transport
    wrapper, so this function tries structured data first and then falls back to
    JSON text or plain text.
    """
    structured = getattr(result, "structured_content", None)
    if structured is not None:
        return as_jsonable(structured)

    content = getattr(result, "content", None)
    if content:
        values = []
        for item in content:
            text = getattr(item, "text", None)
            if text is None:
                values.append(as_jsonable(item))
                continue
            try:
                values.append(json.loads(text))
            except json.JSONDecodeError:
                values.append(text)
        return values[0] if len(values) == 1 else values

    return as_jsonable(result)


async def call_mcp_tool(tool_name: str, arguments: dict[str, Any] | None = None) -> Any:
    """Call one MCP tool through a short-lived client session.

    The orchestration pattern is: connect to the MCP server, initialize the
    session, call a named tool with JSON-like arguments, and normalize the result.
    Keeping this in one helper lets the later agent example focus on tool choice
    instead of connection plumbing.
    """
    async with streamablehttp_client(mcp_url, headers=mcp_headers) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments or {})
            return parse_tool_result(result)


async def list_mcp_tools() -> list[dict[str, Any]]:
    """Discover tool metadata exposed by the MCP server.

    Agents need tool names, descriptions, and schemas before they can decide what
    to call. This mirrors how frameworks such as LangChain or Semantic Kernel
    inspect available tools before planning an action.
    """
    async with streamablehttp_client(mcp_url, headers=mcp_headers) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools = await session.list_tools()
            return [as_jsonable(tool) for tool in tools.tools]

## 3. Discover Available Tools

Tool discovery is important for agents because the model or orchestration layer needs names, descriptions, and schemas before it can decide which tool to call.

In [ ]:
tools = await list_mcp_tools()
display(
    spark.createDataFrame(
        [
            {
                "name": tool.get("name"),
                "description": tool.get("description"),
            }
            for tool in tools
        ]
    )
)

## 4. Call Metadata Tools

These calls are deterministic: they return health information and workshop module metadata. Deterministic tools are useful because they give an agent reliable facts instead of asking the LLM to remember them.

In [ ]:
health = await call_mcp_tool("health")
modules = await call_mcp_tool("list_workshop_modules")
module_9 = await call_mcp_tool("get_module_summary", {"module_id": 9})

print(json.dumps(health, indent=2))
display(spark.createDataFrame(modules))
module_9

## 5. Call Retrieval Tool

The search tool wraps the same knowledge base used by the RAG chatbot. Exposing retrieval as a tool lets an agent decide when it needs additional context before answering.

In [ ]:
sample_query = "What does the MLOps module deploy?"

search_results = await call_mcp_tool(
    "search_workshop_docs", {"query": sample_query, "k": 3}
)
display(spark.createDataFrame(search_results))

## 6. LLM-Planned Agent Step

The final example lets an LLM choose the MCP tool. The notebook gives the model the discovered tool names, descriptions, and input schemas, asks it to return a JSON tool plan, validates that plan, and then executes the selected tool.

This is still a minimal agent loop, but the decision is no longer hard-coded in Python.

In [ ]:
def llm_text(response: Any) -> str:
    """Extract text from common Databricks Foundation Model response shapes."""
    if isinstance(response, dict):
        choices = response.get("choices", [])
        if choices:
            message = choices[0].get("message", {})
            return message.get("content", choices[0].get("text", str(response)))
    return str(response)


def json_from_llm_text(text: str) -> dict[str, Any]:
    """Parse a JSON object even when the model wraps it in a code fence."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not match:
        raise ValueError(f"LLM did not return a JSON object: {text}")
    return json.loads(match.group(0))


def tool_schema_for_planner(tool: dict[str, Any]) -> dict[str, Any]:
    """Keep only tool metadata that helps the LLM choose and call a tool."""
    return {
        "name": tool.get("name"),
        "description": tool.get("description"),
        "input_schema": tool.get("inputSchema")
        or tool.get("input_schema")
        or tool.get("schema"),
    }


def normalize_tool_arguments(
    tool_name: str, arguments: dict[str, Any], user_message: str
) -> dict[str, Any]:
    """Clean up small LLM planning mistakes before calling the MCP server."""
    normalized = dict(arguments)
    if tool_name == "search_workshop_docs":
        normalized.setdefault("query", user_message)
        if "k" in normalized:
            normalized["k"] = int(normalized["k"])
    if tool_name == "get_module_summary" and "module_id" in normalized:
        normalized["module_id"] = int(normalized["module_id"])
    return normalized


def plan_tool_call_with_llm(
    user_message: str, available_tools: list[dict[str, Any]]
) -> dict[str, Any]:
    """Ask the Foundation Model endpoint to choose one MCP tool and arguments.

    The model is not asked to answer the user directly. Its only job is to plan
    one tool invocation using the MCP tool metadata discovered at runtime.
    """
    tool_schemas = [tool_schema_for_planner(tool) for tool in available_tools]
    tool_names = {tool["name"] for tool in tool_schemas}
    messages = [
        {
            "role": "system",
            "content": (
                "You are a tool selection planner for an MCP agent. "
                "Choose exactly one tool from the provided tools. "
                "Return only valid JSON with keys tool_name and arguments. "
                "Do not answer the user. Do not include markdown. "
                "Use search_workshop_docs for workshop questions that need document context. "
                "Use get_module_summary when the user asks for one module by id. "
                "Use list_workshop_modules when the user asks to list modules. "
                "Use health only for app status checks."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Available tools:\n{json.dumps(tool_schemas, indent=2)}\n\n"
                f"User message:\n{user_message}\n\n"
                'Return JSON like: {"tool_name": "search_workshop_docs", "arguments": {"query": "...", "k": 3}}'
            ),
        },
    ]
    response = deploy_client.predict(
        endpoint=llm_endpoint_name,
        inputs={"messages": messages, "temperature": 0.0, "max_tokens": 300},
    )
    plan = json_from_llm_text(llm_text(response))
    tool_name = plan.get("tool_name")
    arguments = plan.get("arguments", {})
    if tool_name not in tool_names:
        raise ValueError(
            f"LLM selected unknown tool {tool_name!r}. Available tools: {sorted(tool_names)}"
        )
    if not isinstance(arguments, dict):
        raise ValueError(f"LLM returned non-dict tool arguments: {arguments!r}")
    arguments = normalize_tool_arguments(tool_name, arguments, user_message)
    return {"tool_name": tool_name, "arguments": arguments, "raw_response": response}


async def llm_agent_step(
    user_message: str, available_tools: list[dict[str, Any]]
) -> dict[str, Any]:
    """Run one LLM-planned MCP tool call and return a readable trace."""
    plan = plan_tool_call_with_llm(user_message, available_tools)
    tool_result = await call_mcp_tool(plan["tool_name"], plan["arguments"])
    return {
        "user_message": user_message,
        "selected_tool": plan["tool_name"],
        "arguments": plan["arguments"],
        "tool_result": tool_result,
    }

In [ ]:
sample_query = "What modules do we have?"

agent_step = await llm_agent_step(sample_query, tools)
print(json.dumps(agent_step, indent=2)[:4000])